# BLSTM Training - Audio & Visual Modalities

Trains two **separate** BiLSTM models, one per modality, predicting 5 OCEAN traits in `[0,1]` (sigmoid + MSE):
- **audio** : `(batch, 15, 128)` VGGish embeddings -> `models/bilstm_audio_tf.keras`
- **visual**: `(batch, 30, 4096)` VGG-Face embeddings -> `models/bilstm_visual_tf.keras`

Architecture per modality: `Bidirectional(LSTM(64))` -> `Dropout(0.3)` -> `Dense(64, ReLU)` -> `Dropout(0.3)` -> `Dense(5, sigmoid)`, Adam lr 1e-4 wd 1e-5, MSE/MAE, batch 32, EarlyStopping on val MAE.

Features z-scored per dimension with **train-only** stats; the same stats are written to `app_prediction/models/norm_stats_{mod}.json` so inference (`personality_detector.py`) normalizes identically.

Val/test annotation zips are password-protected; the password is read from `password.txt` (in the annotations dir) from memory.


In [33]:
import argparse
import gc
import json
import math
import os
import pickle
import random
import sys
import zipfile
import numpy as np
# Absolute path to the root script file
APP_ROOT = os.path.abspath(os.getcwd())
PROJECT_ROOT = os.path.dirname(APP_ROOT)
DEFAULT_FEATURES_ROOT = os.path.join(PROJECT_ROOT, "output")
DEFAULT_ANNOTATIONS_DIR = os.path.join(PROJECT_ROOT, "first-impressions", "annotations")
DEFAULT_MODEL_DIR = os.path.join(APP_ROOT, "models")
DEFAULT_STATS_DIR = os.path.join(APP_ROOT, "app_prediction", "models")
OCEAN = ["extraversion", "neuroticism", "agreeableness", "conscientiousness", "openness"]
SEQ_LEN = {"audio": 15, "visual": 30}
FEAT_DIM = {"audio": 128, "visual": 4096}
MODALITIES = ("audio", "visual")
DEFAULT_EPOCHS = {"audio": 90, "visual": 55}
DEFAULT_HIDDEN = 64
DEFAULT_DROPOUT = 0.3
DEFAULT_BATCH_SIZE = 32
DEFAULT_LR = 1e-4
DEFAULT_WEIGHT_DECAY = 1e-5
DEFAULT_PATIENCE = 10
SEED = 42


In [34]:
DEFAULT_MODEL_DIR

'/mnt/4A3ED7573ED73AA1/aa-kuliah/skripsi/app/models'

In [35]:
def load_labels(annotations_dir, split):
    """Return {video_id: float32 (5,)} for one split.

    train -> train-annotation/annotation_training.pkl (plain file)
    val   -> val-annotation-e.zip  (read in-memory, never unzipped)
    test  -> test-annotation-e.zip (read in-memory, never unzipped)
    """
    if split == "train":
        path = os.path.join(annotations_dir, "train-annotation", "annotation_training.pkl")
        with open(path, "rb") as f:
            ann = pickle.load(f, encoding="latin1")
    elif split == "val":
        path = os.path.join(annotations_dir, "val-annotation-e.zip")
        with _open_zip_pkl(path, "annotation_validation.pkl") as f:
            ann = pickle.load(f, encoding="latin1")
    elif split == "test":
        path = os.path.join(annotations_dir, "test-annotation-e.zip")
        with _open_zip_pkl(path, "annotation_test.pkl") as f:
            ann = pickle.load(f, encoding="latin1")
    else:
        raise ValueError(f"unknown split: {split}")
    mapping = {}
    for trait in OCEAN:
        for clip, score in ann[trait].items():
            stem = clip[:-4] if clip.endswith(".mp4") else clip
            mapping.setdefault(stem, {})[trait] = float(score)

    labels = {}
    for stem, m in mapping.items():
        if all(t in m for t in OCEAN):
            labels[stem] = np.array([m[t] for t in OCEAN], dtype=np.float32)
    return labels

def _open_zip_pkl(zip_path, member):
    """Open a (possibly encrypted) member pkl of an annotation zip.

    The ChaLearn annotation zips are password-protected; the password is
    expected in a ``password.txt`` next to the zip and is read from memory
    (nothing is extracted to disk).
    """
    pwd_path = os.path.join(os.path.dirname(zip_path), "password.txt")
    try:
        with open(pwd_path, "rb") as f:
            pwd = f.read().strip()
    except OSError:
        raise RuntimeError(
            f"{zip_path} is password-protected; put the password in {pwd_path}"
        )
    return _ZipMember(zipfile.ZipFile(zip_path), member, pwd)

class _ZipMember:
    """Context manager closing both the zip and the open member file."""

    def __init__(self, zf, member, pwd):
        self._zip = zf
        self._member = zf.open(member, pwd=pwd)

    def __enter__(self):
        return self._member

    def __exit__(self, *exc):
        self._member.close()
        self._zip.close()


In [36]:
def feature_clips(features_root, mod, split):
    d = os.path.join(features_root, mod, split)
    if not os.path.isdir(d):
        raise FileNotFoundError(f"feature dir not found: {d}")
    return sorted(
        os.path.splitext(f)[0]
        for f in os.listdir(d)
        if f.endswith(".npy")
    )

def load_features(features_root, mod, split, clips, dtype):
    """Stack <root>/<mod>/<split>/<id>.npy into (n, seq, feat) `dtype`.

    Preallocates the output array and fills it file-by-file so peak memory is
    one output array + one file (no list->stack doubling).
    """
    seq_len, feat_dim = SEQ_LEN[mod], FEAT_DIM[mod]
    X = np.empty((len(clips), seq_len, feat_dim), dtype=dtype)
    kept = []
    i = 0
    for stem in clips:
        path = os.path.join(features_root, mod, split, f"{stem}.npy")
        try:
            arr = np.load(path)
        except (OSError, ValueError):
            continue
        if arr.shape != (seq_len, feat_dim) or not np.isfinite(arr).all():
            continue
        X[i] = arr
        kept.append(stem)
        i += 1
    if i == 0:
        raise RuntimeError(f"no usable {mod} features in {os.path.join(features_root, mod, split)}")
    return X[:i], kept

def mem_available_bytes():
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemAvailable:"):
                    return int(line.split()[1]) * 1024
    except OSError:
        pass
    return None

def pick_dtype(args, n_clips_by_split, mod):
    if args.dtype != "auto":
        return np.dtype(args.dtype)
    total_elems = sum(n_clips_by_split.values()) * SEQ_LEN[mod] * FEAT_DIM[mod]
    need32 = total_elems * 4 + 1_500_000_000  # float32 data + TF runtime
    avail = mem_available_bytes()
    if avail is None or need32 <= avail * 1.3:
        return np.dtype(np.float32)
    return np.dtype(np.float16)


In [37]:
def build_model(tf, seq_len, feat_dim, hidden, dropout, lr, weight_decay):
    inp = tf.keras.Input(shape=(seq_len, feat_dim))
    x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(hidden))(inp)
    x = tf.keras.layers.Dropout(dropout)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    out = tf.keras.layers.Dense(5, activation="sigmoid", name="ocean_output")(x)
    model = tf.keras.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr, weight_decay=weight_decay),
        loss="mse",
        metrics=["mae"],
    )
    return model

def make_sequence_class(tf):
    class _Seq(tf.keras.utils.Sequence):
        def __init__(self, X, y, batch_size, shuffle, seed=SEED):
            super().__init__()
            self.X = X
            self.y = y
            self.batch_size = batch_size
            self.shuffle = shuffle
            self.rng = np.random.default_rng(seed)
            self.indices = np.arange(len(X))
            if shuffle:
                self.rng.shuffle(self.indices)

        def __len__(self):
            return math.ceil(len(self.X) / self.batch_size)

        def __getitem__(self, i):
            idx = self.indices[i * self.batch_size:(i + 1) * self.batch_size]
            xb = self.X[idx]
            if xb.dtype != np.float32:
                xb = xb.astype(np.float32)
            return xb, self.y[idx]

        def on_epoch_end(self):
            if self.shuffle:
                self.rng.shuffle(self.indices)

    return _Seq


In [38]:
def regression_metrics(y_true, y_pred):
    yt = y_true.astype(np.float64)
    yp = y_pred.astype(np.float64)
    err = yp - yt
    mae = np.mean(np.abs(err), axis=0)
    rmse = np.sqrt(np.mean(err ** 2, axis=0))
    ss_res = np.sum(err ** 2, axis=0)
    ss_tot = np.sum((yt - yt.mean(axis=0)) ** 2, axis=0)
    r2 = np.where(ss_tot > 0, 1.0 - ss_res / np.maximum(ss_tot, 1e-12), 0.0)
    return {
        "overall": {
            "mae": float(mae.mean()),
            "rmse": float(np.sqrt(np.mean(err ** 2))),
            "r2": float(r2.mean()),
        },
        "per_trait": {
            t: {"mae": float(mae[i]), "rmse": float(rmse[i]), "r2": float(r2[i])}
            for i, t in enumerate(OCEAN)
        },
    }

def print_metrics(title, m):
    print(f"{title}:")
    print(f"  {'trait':<18}{'MAE':>10}{'RMSE':>10}{'R2':>10}")
    for t in OCEAN:
        d = m["per_trait"][t]
        print(f"  {t:<18}{d['mae']:10.4f}{d['rmse']:10.4f}{d['r2']:10.4f}")
    o = m["overall"]
    print(f"  {'overall':<18}{o['mae']:10.4f}{o['rmse']:10.4f}{o['r2']:10.4f}")


In [39]:
def train_modality(mod, args, tf):
    seq_len, feat_dim = SEQ_LEN[mod], FEAT_DIM[mod]
    epochs = args.epochs if args.epochs is not None else DEFAULT_EPOCHS[mod]

    print(f"\n=== {mod} ===")
    labels = {"train": load_labels(args.annotations_dir, "train"),
              "val": load_labels(args.annotations_dir, "val")}
    splits_needed = ["train", "val"] + (["test"] if args.eval_test else [])
    for s in splits_needed:
        if s not in labels:
            labels[s] = load_labels(args.annotations_dir, s)

    clips = {}
    for s in splits_needed:
        c = [x for x in feature_clips(args.features_root, mod, s) if x in labels[s]]
        if args.limit:
            c = c[: args.limit]
        clips[s] = c
        print(f"  clips {s}: {len(c)}")

    dtype = pick_dtype(args, {s: len(clips[s]) for s in splits_needed}, mod)
    print(f"  storage dtype: {dtype}")

    data = {}
    for s in ("train", "val"):
        data[s] = load_features(args.features_root, mod, s, clips[s], dtype)
    Xtr, kept_tr = data["train"]
    Xva, kept_va = data["val"]
    ytr = np.stack([labels["train"][c] for c in kept_tr])
    yva = np.stack([labels["val"][c] for c in kept_va])

    mean = np.mean(Xtr, axis=(0, 1), dtype=np.float64)
    std = np.std(Xtr, axis=(0, 1), dtype=np.float64) + 1e-8
    Xtr -= mean
    Xtr /= std
    Xva -= mean
    Xva /= std

    os.makedirs(args.stats_dir, exist_ok=True)
    stats_path = os.path.join(args.stats_dir, f"norm_stats_{mod}.json")
    with open(stats_path, "w") as f:
        json.dump({"mean": mean.tolist(), "std": std.tolist()}, f)
    print(f"  saved {stats_path}")

    model = build_model(tf, seq_len, feat_dim, args.hidden, args.dropout,
                        args.learning_rate, args.weight_decay)
    model.summary()

    Seq = make_sequence_class(tf)
    train_ds = Seq(Xtr, ytr, args.batch_size, shuffle=True)
    val_ds = Seq(Xva, yva, args.batch_size, shuffle=False)
    es = tf.keras.callbacks.EarlyStopping(
        monitor="val_mae", patience=args.patience,
        restore_best_weights=True, verbose=1,
    )
    history = model.fit(
        train_ds, validation_data=val_ds, epochs=epochs,
        callbacks=[es], verbose=2,
    ).history

    y_pred = model.predict(val_ds, verbose=0)
    val_metrics = regression_metrics(yva, y_pred)
    best_epoch = int(np.argmin(history["val_mae"])) + 1
    best_val_mae = float(np.min(history["val_mae"]))
    print(f"  best epoch {best_epoch}/{epochs}  val MAE {best_val_mae:.4f}")
    print_metrics(f"  {mod} val metrics", val_metrics)

    os.makedirs(args.output_dir, exist_ok=True)
    model_path = os.path.join(args.output_dir, f"bilstm_{mod}_tf.keras")
    model.save(model_path)
    print(f"  saved {model_path}")

    meta = {
        "modality": mod,
        "traits": OCEAN,
        "config": {
            "seq_len": seq_len,
            "feat_dim": feat_dim,
            "hidden": args.hidden,
            "dropout": args.dropout,
            "batch_size": args.batch_size,
            "learning_rate": args.learning_rate,
            "weight_decay": args.weight_decay,
            "epochs": epochs,
            "patience": args.patience,
            "seed": SEED,
            "storage_dtype": str(dtype),
        },
        "data": {
            "train_clips": len(kept_tr),
            "val_clips": len(kept_va),
            "features_root": args.features_root,
        },
        "norm_stats": stats_path,
        "best_epoch": best_epoch,
        "best_val_mae": best_val_mae,
        "val_metrics": val_metrics,
        "history": {k: [float(v) for v in vals] for k, vals in history.items()},
    }

    if args.eval_test:
        Xte, kept_te = load_features(args.features_root, mod, "test",
                                     clips["test"], dtype)
        yte = np.stack([labels["test"][c] for c in kept_te])
        Xte -= mean
        Xte /= std
        test_ds = Seq(Xte, yte, args.batch_size, shuffle=False)
        test_metrics = regression_metrics(yte, model.predict(test_ds, verbose=0))
        print_metrics(f"  {mod} test metrics", test_metrics)
        meta["test_metrics"] = test_metrics
        meta["data"]["test_clips"] = len(kept_te)

    history_path = os.path.join(args.output_dir, f"bilstm_{mod}_tf_history.json")
    with open(history_path, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"  saved {history_path}")

    del Xtr, Xva, ytr, yva, train_ds, val_ds, model
    gc.collect()
    return meta


In [40]:
def prepare_tf():
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
    import tensorflow as tf

    gpus = tf.config.list_physical_devices("GPU")
    for g in gpus:
        try:
            tf.config.experimental.set_memory_growth(g, True)
        except Exception:
            pass
    # mixed_float16 only where it helps (GPU); CPU training stays float32
    try:
        if gpus:
            tf.keras.mixed_precision.set_global_policy("mixed_float16")
        else:
            tf.keras.mixed_precision.set_global_policy("float32")
    except Exception:
        tf.keras.mixed_precision.set_global_policy("float32")
    tf.keras.utils.set_random_seed(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    print("TF GPUs:", gpus or "none (CPU training)")
    return tf


## Configuration

Edit `modality`, `epochs`, `limit`/`dtype` as needed. Defaults train both modalities for the full proven schedule (audio 90 epochs, visual 55) with float32 storage; `dtype="auto"` falls back to float16 for the visual set on low-RAM machines.


In [41]:
import types

config = types.SimpleNamespace(
    modality="both",            # "audio" | "visual" | "both"
    features_root=DEFAULT_FEATURES_ROOT,
    annotations_dir=DEFAULT_ANNOTATIONS_DIR,
    output_dir=DEFAULT_MODEL_DIR,
    stats_dir=DEFAULT_STATS_DIR,
    epochs=None,                # None -> per-modality default (audio 90, visual 55)
    batch_size=DEFAULT_BATCH_SIZE,
    learning_rate=DEFAULT_LR,
    weight_decay=DEFAULT_WEIGHT_DECAY,
    hidden=DEFAULT_HIDDEN,
    dropout=DEFAULT_DROPOUT,
    patience=DEFAULT_PATIENCE,
    limit=None,                 # max clips per split (e.g. 128 for a smoke run)
    eval_test=False,            # also evaluate on the test split
    dtype="auto",               # "auto" | "float32" | "float16"
)


In [42]:
tf = prepare_tf()

results = []
for mod in (MODALITIES if config.modality == "both" else (config.modality,)):
    results.append(train_modality(mod, config, tf))
    tf.keras.backend.clear_session()
    gc.collect()

print("\n=== summary ===")
for r in results:
    o = r["val_metrics"]["overall"]
    test = ""
    if r.get("test_metrics"):
        t = r["test_metrics"]["overall"]
        test = f"  test MAE {t['mae']:.4f} RMSE {t['rmse']:.4f} R2 {t['r2']:.4f}"
    print(f"{r['modality']:<8} clips {r['data']['train_clips']}/{r['data']['val_clips']} "
          f"| best epoch {r['best_epoch']} | val MAE {o['mae']:.4f} RMSE {o['rmse']:.4f} R2 {o['r2']:.4f}"
          f"{test} | models/bilstm_{r['modality']}_tf.keras")


TF GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

=== audio ===
  clips train: 5962
  clips val: 1987
  storage dtype: float32
  saved /mnt/4A3ED7573ED73AA1/aa-kuliah/skripsi/app/app_prediction/models/norm_stats_audio.json


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 15, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ocean_output (Dense)            │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 107,397 (419.52 KB)

 Trainable params: 107,397 (419.52 KB)

...

 Total params: 2,139,013 (8.16 MB)

 Trainable params: 2,139,013 (8.16 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/55
187/187 - 6s - 34ms/step - loss: 0.0324 - mae: 0.1440 - val_loss: 0.0207 - val_mae: 0.1149
Epoch 2/55
187/187 - 3s - 15ms/step - loss: 0.0238 - mae: 0.1232 - val_loss: 0.0187 - val_mae: 0.1091
Epoch 3/55
187/187 - 3s - 14ms/step - loss: 0.0195 - mae: 0.1109 - val_loss: 0.0172 - val_mae: 0.1047
Epoch 4/55
187/187 - 3s - 14ms/step - loss: 0.0167 - mae: 0.1028 - val_loss: 0.0163 - val_mae: 0.1020
Epoch 5/55
187/187 - 3s - 15ms/step - loss: 0.0143 - mae: 0.0947 - val_loss: 0.0156 - val_mae: 0.0996
Epoch 6/55
187/187 - 3s - 15ms/step - loss: 0.0126 - mae: 0.0889 - val_loss: 0.0151 - val_mae: 0.0978
Epoch 7/55
187/187 - 3s - 14ms/step - loss: 0.0112 - mae: 0.0840 - val_loss: 0.0147 - val_mae: 0.0964
Epoch 8/55
187/187 - 3s - 14ms/step - loss: 0.0102 - mae: 0.0799 - val_loss: 0.0143 - val_mae: 0.0951
Epoch 9/55
187/187 - 3s - 14ms/step - loss: 0.0092 - mae: 0.0761 - val_loss: 0.0142 - val_mae: 0.0945
Epoch 10/55
187/187 - 3s - 14ms/step - loss: 0.0085 - mae: 0.0729 - val_loss: 0.01

KeyboardInterrupt: 

## Results

Saved artifacts (in `models/` and `app_prediction/models/`):
- `bilstm_audio_tf.keras`, `bilstm_visual_tf.keras` - Keras models
- `bilstm_{mod}_tf_history.json` - config, early-stopped epoch, val/test MAE/RMSE/R2 (overall + per trait), training history
- `norm_stats_{mod}.json` - train-only per-dim z-score mean/std
